# Targeted Attack Training for Demo Phrase

## Overview
This notebook demonstrates the process of training a **Carlini-Wagner (CW) targeted attack** specifically for the demo phrase. 
The goal is to generate a perturbation that forces Whisper to output the target string:
**"This is a Demo - aai590"**

This ensures the live demo can reliably showcase the injection capability.

In [ ]:
import torch
import whisper
import numpy as np
import soundfile as sf
import librosa
import os
import json
from src.attacks.cw import CWAttack
from src.attacks.utils import compute_snr

## 1. Configuration & Hyperparameters
We define the target phrase and attack parameters.

In [ ]:
# Configuration
TARGET_PHRASE = "This is a Demo - aai590"
MODEL_SIZE = "base"

# CW Attack Parameters
# These are tuned for a clean, high-quality demo voice
EPSILON = 0.05    # Perturbation magnitude (L2 or Linf)
ITERATIONS = 2000 # Optimization steps
LEARNING_RATE = 0.01
TARGETED = True   # We want the model to output the specific phrase
CONFIDENCE_THRESHOLD = 0.0 # Lower this if confidence is low

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2. Load Model
Load Whisper model in fp16 for faster inference if on CUDA.

In [ ]:
print("Loading Whisper model...")
model = whisper.load_model(MODEL_SIZE, device=DEVICE).to(DEVICE).half()
print("Model loaded successfully.")

## 3. Define CW Attack Wrapper
We need a function that takes a clean audio sample, optimizes a perturbation, and returns the adversarial audio and metrics.

In [ ]:
def generate_targeted_attack(clean_audio, model, target_phrase, params):
    """
    Generates a targeted adversarial audio sample.
    
    Args:
        clean_audio (np.array): Audio waveform [-1, 1]
        model: Whisper model
        target_phrase (str): Desired output text
        params (dict): CW attack parameters
        
    Returns:
        adv_audio (np.array): Adversarial waveform
        transcript (str): Model prediction on adv_audio
        snr_db (float): Signal-to-Noise Ratio of attack
    """
    # Convert to tensor (Whisper expects float32)
    clean_tensor = torch.from_numpy(clean_audio.astype(np.float32)).unsqueeze(0).to(DEVICE)
    clean_tensor.requires_grad = True
    
    # Initialize perturbation
    adv_tensor = clean_tensor.clone()
    
    # Initialize Attack Object
    cw = CWAttack(
        model=model,
        confidence=CONFIDENCE_THRESHOLD,
        epsilon=params['EPSILON'],
        learning_rate=params['LEARNING_RATE'],
        max_iter=params['ITERATIONS'],
        device=DEVICE,
        targeted=params['TARGETED']
    )
    
    # Run Optimization
    print(f"  Starting optimization... (Target: '{target_phrase}')")
    adv_tensor, loss_history = cw.attack(adv_tensor)
    
    # Clip to valid audio range [-1, 1]
    adv_tensor = torch.clamp(adv_tensor, -1.0, 1.0)
    
    # Convert back to numpy
    adv_audio = adv_tensor.squeeze().cpu().numpy().astype(np.float32)
    
    # Transcribe
    result = model.transcribe(adv_audio)
    transcript = result["text"].strip()
    
    # Calculate SNR
    snr_db = compute_snr(clean_audio, adv_audio)
    
    return adv_audio, transcript, snr_db

## 4. Run Attack on Demo Voice
Here we would load a specific demo voice sample. For this script, we will generate a synthetic test sample (silence + a tone) or use a pre-recorded file if available.

Since we cannot record here, we will use a 'dummy' audio signal (0.0) to demonstrate the *mechanism*, or simply explain that this step requires the user to provide their own voice sample file.

### Simulation
In a real run, you would load `demo_assets/my_voice_generic.wav`.

In [ ]:
# Simulation: Generating a short test signal
# Real operation: Load your own voice file
print("Generating synthetic test signal for demonstration...")
sample_rate = 16000
duration = 3.0 # seconds
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# Create a simple tone (400Hz sine wave) to act as 'clean' input
clean_audio = 0.5 * np.sin(2 * np.pi * 400 * t).astype(np.float32)

print(f"Clean Audio Shape: {clean_audio.shape}")
print(f"Clean Audio Min/Max: {clean_audio.min():.3f}, {clean_audio.max():.3f}")

# Run Attack
attack_params = {
    'EPSILON': EPSILON,
    'LEARNING_RATE': LEARNING_RATE,
    'ITERATIONS': ITERATIONS,
    'TARGETED': TARGETED
}

adv_audio, transcript, snr = generate_targeted_attack(
    clean_audio, model, TARGET_PHRASE, attack_params
)

## 5. Results Analysis
Analyze the attack success.

In [ ]:
print("\n=== Attack Results ===")
print(f"Target Phrase:       {TARGET_PHRASE}")
print(f"Predicted Transcript: {transcript}")
print(f"Success:              {TARGET_PHRASE in transcript}")
print(f"SNR:                  {snr:.2f} dB")

# Visualize Loss
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(loss_history)
plt.title("CW Optimization Loss")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

## 6. Save the Trained Perturbation
Save the adversarial example to disk for use in the live demo.

In [ ]:
SAVE_PATH = "demo_assets/targeted_perturbation.pt"

output = {
    "target_phrase": TARGET_PHRASE,
    "audio_sample": adv_audio,
    "params": attack_params,
    "metrics": {
        "transcript": transcript,
        "snr": snr,
        "success": TARGET_PHRASE in transcript
    }
}

# Create directory if needed
os.makedirs("demo_assets", exist_ok=True)

torch.save(output, SAVE_PATH)
print(f"Saved adversarial example to: {SAVE_PATH}")

## 7. Troubleshooting & Tuning

If the attack fails (transcript != target phrase):

1.  **Increase Epsilon**: Try `EPSILON = 0.08`. Larger perturbation is easier to detect by human ears but easier for model to force.
2.  **Increase Iterations**: Try `ITERATIONS = 3000`. More optimization steps usually converge to a better solution.
3.  **Check Confidence Threshold**: Lower `CONFIDENCE_THRESHOLD` to `0.0` or `-0.5`. Lower confidence means the attack allows the model to be 'less sure' about its answer, which is helpful for CW.
4.  **Target Phrase Length**: Make sure the target phrase is short (e.g., "Demo aai590") if Whisper tends to hallucinate endings.